# Neural Identifier Training with Particle Filters - Differential Drive Mobile Robot

In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
# ============================================================
# 1) True nonlinear system (Differential Drive Mobile Robot)
# ============================================================
def plant_dynamics(x, u, L=0.5, friction_coeff=0.1, slip_factor=0.05):
    """
    Continuous dynamics for differential drive mobile robot: x = [x_pos, y_pos, theta]. 
    Returns x_dot.
    
    The differential drive robot equations:
    dx/dt = v * cos(θ) 
    dy/dt = v * sin(θ)
    dθ/dt = ω
    
    where:
    v = (v_r + v_l) / 2  (linear velocity)
    ω = (v_r - v_l) / L  (angular velocity)
    L = wheelbase distance
    """
    x_pos, y_pos, theta = x
    v_l, v_r = u  # left and right wheel velocities
    
    # Add realistic wheel slip effects
    v_l_actual = v_l * (1 - slip_factor * np.random.randn())
    v_r_actual = v_r * (1 - slip_factor * np.random.randn())
    
    # Compute linear and angular velocities
    v = (v_r_actual + v_l_actual) / 2.0
    omega = (v_r_actual - v_l_actual) / L
    
    # Add friction effects (velocity-dependent)
    v_friction = v * (1 - friction_coeff * np.abs(v))
    omega_friction = omega * (1 - friction_coeff * np.abs(omega))
    
    # Robot kinematics
    x_dot = v_friction * np.cos(theta)
    y_dot = v_friction * np.sin(theta)
    theta_dot = omega_friction
    
    return np.array([x_dot, y_dot, theta_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='mixed', process_noise_std=0.05, 
          terrain_roughness=0.02, sensor_bias=[0.0, 0.0, 0.0]):
    """
    One Euler step of the discrete plant with realistic mobile robot disturbances.
    
    Args:
        x_k: current state [x, y, theta]
        u_k: control input [v_left, v_right] 
        dt: time step
        process_noise_type: type of noise ('mixed', 'gaussian', 'laplacian')
        process_noise_std: standard deviation of process noise
        terrain_roughness: terrain-induced disturbances
        sensor_bias: systematic biases in measurements
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Realistic mobile robot disturbances
    
    # 1. Terrain-induced disturbances (position-dependent)
    terrain_noise = terrain_roughness * np.array([
        np.sin(0.5 * x_kp1[0]) * np.random.randn(),  # x-direction terrain variation
        np.cos(0.3 * x_kp1[1]) * np.random.randn(),  # y-direction terrain variation  
        0.1 * np.sin(x_kp1[2]) * np.random.randn()   # angular disturbance from terrain
    ])
    
    # 2. Velocity-dependent noise (increases with speed)
    velocity_magnitude = np.linalg.norm(u_k)
    velocity_noise_factor = 1 + 0.2 * velocity_magnitude
    
    # 3. Mixed process noise (combination of different noise types)
    if process_noise_type == 'mixed':
        # Gaussian component (main noise)
        gaussian_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
        # Impulse noise (occasional large disturbances)
        impulse_prob = 0.02  # 2% chance of impulse noise
        impulse_noise = np.zeros_like(x_kp1)
        if np.random.rand() < impulse_prob:
            impulse_noise = np.random.normal(0, process_noise_std * 5, size=x_kp1.shape)
        # Laplacian component (heavy-tailed noise)
        laplacian_noise = np.random.laplace(0, process_noise_std * 0.3, size=x_kp1.shape)
        
        total_noise = gaussian_noise + impulse_noise + laplacian_noise
    elif process_noise_type == 'laplacian':
        total_noise = np.random.laplace(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std * velocity_noise_factor
        total_noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        total_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    
    # 4. Systematic biases (drift, calibration errors)
    bias_noise = np.array(sensor_bias) * dt
    
    # 5. Wheel encoder quantization effects
    encoder_resolution = 0.001  # 1mm resolution
    quantization_noise = encoder_resolution * (np.random.rand(3) - 0.5)
    
    # Combine all disturbances
    x_kp1 += terrain_noise + total_noise + bias_noise + quantization_noise
    
    # 6. Angle wrapping for realistic behavior
    x_kp1[2] = np.arctan2(np.sin(x_kp1[2]), np.cos(x_kp1[2]))  # wrap angle to [-π, π]
    
    return x_kp1

def generate_realistic_trajectory(t, trajectory_type='straight'):
    """
    Generate realistic control inputs for mobile robot.
    
    Args:
        t: time value
        trajectory_type: 'straight', 'circle', 'figure8', 'mixed', 'obstacle_avoidance'
    
    Returns:
        u: [v_left, v_right] wheel velocities
    """
    if trajectory_type == 'straight':
        # Straight line with small variations
        v_base = 1.0 + 0.2 * np.sin(0.5 * t)
        return np.array([v_base, v_base])
    
    elif trajectory_type == 'circle':
        # Circular motion
        v_l = 1.0 + 0.1 * np.sin(t)
        v_r = 1.5 + 0.1 * np.cos(t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'figure8':
        # Figure-8 pattern
        v_l = 1.0 + 0.8 * np.sin(0.5 * t)
        v_r = 1.0 - 0.8 * np.sin(0.5 * t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'obstacle_avoidance':
        # Obstacle avoidance maneuvers
        base_speed = 1.2
        avoidance_maneuver = 0.5 * np.sin(2 * t) * np.exp(-0.1 * t)
        v_l = base_speed + avoidance_maneuver
        v_r = base_speed - avoidance_maneuver
        return np.array([v_l, v_r])
    
    else:  # 'mixed' - combination of different behaviors
        # Mixed trajectory with different phases
        phase = (t % 20.0) / 20.0  # 20-second cycles
        
        if phase < 0.3:  # Straight motion
            v_base = 1.5
            return np.array([v_base, v_base])
        elif phase < 0.6:  # Turning
            v_l = 0.8
            v_r = 1.8
            return np.array([v_l, v_r])
        elif phase < 0.8:  # Reverse
            v_base = -0.5
            return np.array([v_base, v_base])
        else:  # Complex maneuver
            v_l = 1.0 + 0.5 * np.sin(10 * t)
            v_r = 1.0 + 0.5 * np.cos(10 * t)
            return np.array([v_l, v_r])

In [3]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    # z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input=None):
    """
    Features for a 3-state mobile robot system with control inputs:
    x = [x_pos, y_pos, theta], u = [v_left, v_right]
    
    z = [S(x), S(y), S(θ), S(x)S(y), S(x)S(θ), S(y)S(θ), 
         S(x)^2, S(y)^2, S(θ)^2, cos(θ), sin(θ), 
         S(v_l), S(v_r), S(v_l)S(v_r), x, y, 1]
    """
    s_x = sigmoidal(x_est[0])     # x position
    s_y = sigmoidal(x_est[1])     # y position  
    s_theta = sigmoidal(x_est[2]) # orientation
    
    # Basic features
    features = [
        s_x, s_y, s_theta,                    # Individual sigmoid terms
        s_x*s_y, s_x*s_theta, s_y*s_theta,   # Cross terms
        s_x**2, s_y**2, s_theta**2,          # Quadratic terms
        np.cos(x_est[2]), np.sin(x_est[2]),  # Trigonometric terms (important for robot)
    ]
    
    # Add control input features if available
    if u_input is not None and len(u_input) >= 2:
        s_vl = sigmoidal(u_input[0])  # left wheel velocity
        s_vr = sigmoidal(u_input[1])  # right wheel velocity
        features.extend([
            s_vl, s_vr,                       # Control sigmoid terms
            s_vl * s_vr,                      # Control cross term
        ])
    else:
        # Add zero placeholders if no control input
        features.extend([0.0, 0.0, 0.0])
    
    # Add direct state terms and bias
    features.extend([
        x_est[0], x_est[1],                   # Direct position terms
        1.0                                   # Bias term
    ])
    
    return np.array(features)


def RHONN_predict(x_state_for_z, w_neuron, u_input=None):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z, u_input)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [4]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [5]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=None, R_std=None, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Handle Q_std (process noise) - can be scalar or list/array per neuron
        if Q_std is None:
            self.Q_std = [0.05] * num_neurons
        elif isinstance(Q_std, (int, float)):
            self.Q_std = [Q_std] * num_neurons
        else:
            self.Q_std = list(Q_std) if len(Q_std) == num_neurons else [0.05] * num_neurons
            
        # Handle R_std (measurement noise) - can be scalar or list/array per neuron  
        if R_std is None:
            self.R_std = [0.1] * num_neurons
        elif isinstance(R_std, (int, float)):
            self.R_std = [R_std] * num_neurons
        else:
            self.R_std = list(R_std) if len(R_std) == num_neurons else [0.1] * num_neurons
            
        # Compute R_var for each neuron
        self.R_var = [r**2 for r in self.R_std]
        
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : filter's own estimate at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        u_input: control input at time k (for mobile robot)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)
        z = construct_z_vector(x_state_for_z, u_input)  # (num_features,)

        # 1) Predict: random walk on weights with state-specific Q_std
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std[i]

        # 2) Update: importance weights with Gaussian likelihood using state-specific R_var
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability using state-specific R_var
            ll = -0.5 * (innov**2) / self.R_var[i]
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

    def get_parameters_info(self):
        """Return information about the PF parameters for each state."""
        state_names = ['x', 'y', 'theta']
        info = {}
        for i in range(self.num_neurons):
            name = state_names[i] if i < len(state_names) else f'state_{i}'
            info[name] = {
                'Q_std': self.Q_std[i],
                'R_std': self.R_std[i],
                'R_var': self.R_var[i]
            }
        return info

In [6]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (filter's own estimate for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

## 🔧 Parameter Optimization via Differential Evolution (DE)

We'll optimize filter hyperparameters using **Differential Evolution**, a population-based global optimizer well-suited for noisy, non-convex error landscapes.

### Why Differential Evolution?
- Gradient-free (robust to non-differentiable effects from resampling / stochastic noise)
- Maintains population diversity (helps avoid premature convergence)
- Simple control (mutation factor F, crossover rate CR)

### Filters & Tunable Parameters
We optimize (separately) the following continuous parameters:

| Filter | Parameters (vector order) | Bounds |
|--------|---------------------------|--------|
| EKF | [Q_init, R_init, P_init, eta] | [1e-6–1e-2, 1e-5–1e-1, 0.1–10, 0.1–1.2] |
| UKF | [Q_init, R_init, P_init, eta, alpha] | same first 4 + alpha: 1e-4–0.5 |
| PF  | [Q_x, Q_y, Q_theta, R_x, R_y, R_theta, ess_ratio] | Q/R: 1e-3–1.0, ess_ratio: 0.3–0.9 |

Number of particles (500) is fixed as requested.

### Objective Function
For each candidate θ:
1. Run a shortened simulation horizon (e.g. reduced n_steps) for speed
2. Compute total MSE = Σ(MSE_x + MSE_y + MSE_theta)
3. Return total MSE (lower is better)

### Computational Strategy
- Use moderate population size (pop = 8–12 × dim)
- Early stop if no improvement for several generations
- Clamp + log-safe handling for pathological candidates

After optimization we re-run the full simulation using best parameter sets.

Next cell: implementation of a lightweight Differential Evolution routine and parameter search wrappers.

In [7]:
import math, time

# ============================================================
# Differential Evolution Optimizer (lightweight)
# ============================================================
def differential_evolution(objective, bounds, pop_factor=10, F=0.7, CR=0.9, generations=30, seed=None, tol=1e-6, stall_generations=8):
    if seed is not None:
        np.random.seed(seed)
    dim = len(bounds)
    pop_size = max(pop_factor * dim, 4)
    # Initialize population uniformly inside bounds
    pop = np.array([
        [np.random.uniform(low, high) for (low, high) in bounds]
        for _ in range(pop_size)
    ])
    scores = np.array([objective(ind) for ind in pop])
    best_idx = int(np.argmin(scores))
    best = pop[best_idx].copy()
    best_score = scores[best_idx]
    no_improve = 0
    history = [(0, best_score)]

    for gen in range(1, generations+1):
        for i in range(pop_size):
            # Mutation: select 3 distinct other indices
            idxs = [idx for idx in range(pop_size) if idx != i]
            a, b, c = pop[np.random.choice(idxs, 3, replace=False)]
            mutant = a + F * (b - c)
            # Crossover
            trial = pop[i].copy()
            j_rand = np.random.randint(0, dim)
            for j in range(dim):
                if np.random.rand() < CR or j == j_rand:
                    trial[j] = mutant[j]
            # Clamp to bounds
            for j, (low, high) in enumerate(bounds):
                if trial[j] < low: trial[j] = low
                if trial[j] > high: trial[j] = high
            # Evaluate
            trial_score = objective(trial)
            if trial_score < scores[i]:
                pop[i] = trial
                scores[i] = trial_score
                if trial_score < best_score - tol:
                    best_score = trial_score
                    best = trial.copy()
        if best_score < history[-1][1] - tol:
            no_improve = 0
        else:
            no_improve += 1
        history.append((gen, best_score))
        if no_improve >= stall_generations:
            break
    return {'best_params': best, 'best_score': best_score, 'history': history}

# ============================================================
# Objective helpers: short-horizon simulation for each filter
# ============================================================

def short_sim_prepare(common_initial_weights, num_neurons, num_features):
    # Provide fresh copies of initial weights per optimization call
    return [np.copy(w) for w in common_initial_weights]

SHORT_STEPS = 400  # reduced horizon for speed

# Control schedule reused (figure8 default). We'll precompute controls for speed.
precomputed_u = None

def ensure_precomputed_controls(dt, trajectory_type='figure8'):
    global precomputed_u
    if precomputed_u is None or len(precomputed_u) != SHORT_STEPS:
        precomputed_u = []
        for k in range(SHORT_STEPS):
            t_current = k * dt
            precomputed_u.append(generate_realistic_trajectory(t_current, trajectory_type))
        precomputed_u = np.array(precomputed_u)
    return precomputed_u

# Shared small plant wrapper for objective

def run_short_sim_EKF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias):
    Q_init, R_init, P_init, eta = params
    num_neurons = 3
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    ekf_local = EKF_RHONN_Trainer(num_neurons, num_features, initial_weights=weights_init, Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta)
    x_true = np.zeros((SHORT_STEPS, 3))
    x_hat = np.zeros((SHORT_STEPS, 3))
    # initial states already zero
    controls = ensure_precomputed_controls(dt)
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        ekf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        x_state_z = np.copy(x_hat[k])
        x_state_z[0] = x_hat[k][0]
        x_hat[k+1, 0] = RHONN_predict(x_state_z, ekf_local.weights[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, ekf_local.weights[1], u_k)
        x_hat[k+1, 2] = RHONN_predict(x_state_z, ekf_local.weights[2], u_k)
    # total MSE
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)+np.mean(err[:,2]**2)


def run_short_sim_UKF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias):
    Q_init, R_init, P_init, eta, alpha = params
    num_neurons = 3
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    ukf_local = UKF_RHONN_Trainer(num_neurons, num_features, initial_weights=weights_init, Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta, alpha=alpha, beta=2.0)
    x_true = np.zeros((SHORT_STEPS, 3))
    x_hat = np.zeros((SHORT_STEPS, 3))
    controls = ensure_precomputed_controls(dt)
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        ukf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        x_state_z = np.copy(x_hat[k])
        x_state_z[0] = x_hat[k][0]
        x_hat[k+1, 0] = RHONN_predict(x_state_z, ukf_local.weights[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, ukf_local.weights[1], u_k)
        x_hat[k+1, 2] = RHONN_predict(x_state_z, ukf_local.weights[2], u_k)
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)+np.mean(err[:,2]**2)


def run_short_sim_PF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias, n_particles_fixed=500):
    Qx, Qy, Qth, Rx, Ry, Rth, ess_ratio = params
    num_neurons = 3
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    pf_local = PF_RHONN_Trainer(num_neurons, num_features, n_particles=n_particles_fixed, initial_weights=weights_init, Q_std=[Qx,Qy,Qth], R_std=[Rx,Ry,Rth], ess_threshold=n_particles_fixed*ess_ratio)
    # Force identical initialization
    for i in range(num_neurons):
        pf_local.particles[i] = np.tile(weights_init[i], (pf_local.n_particles,1))
        pf_local.weights_pf[i] = np.ones(pf_local.n_particles)/pf_local.n_particles
    x_true = np.zeros((SHORT_STEPS, 3))
    x_hat = np.zeros((SHORT_STEPS, 3))
    controls = ensure_precomputed_controls(dt)
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        pf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        pf_w = pf_local.get_estimate()
        x_state_z = np.copy(x_hat[k])
        x_state_z[0] = x_hat[k][0]
        x_hat[k+1, 0] = RHONN_predict(x_state_z, pf_w[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, pf_w[1], u_k)
        x_hat[k+1, 2] = RHONN_predict(x_state_z, pf_w[2], u_k)
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)+np.mean(err[:,2]**2)

# ============================================================
# Wrapper objectives with logging & penalty for instability
# ============================================================

def make_objective(filter_name, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias):
    def obj(params):
        try:
            if filter_name=='EKF':
                return run_short_sim_EKF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
            elif filter_name=='UKF':
                return run_short_sim_UKF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
            else:
                return run_short_sim_PF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
        except Exception as e:
            # Heavy penalty if something blows up
            return 1e6
    return obj

# ============================================================
# Launch optimization (set RUN_DE=True to execute)
# ============================================================
RUN_DE = True  # toggle to False to skip

# Define common_initial_weights before use
import numpy as np
num_neurons = 3
num_features = 6  # Adjust this if your RHONN uses a different number of features
common_initial_weights = [np.zeros(num_features) for _ in range(num_neurons)]

if RUN_DE:
    print("\n[DE] Starting hyperparameter optimization (short horizon)...")
    start_total = time.time()

    # Define missing variables with example/default values
    dt = 0.05  # time step (seconds)
    process_noise_type = 'gaussian'  # or another type your plant() supports
    process_noise_std = 0.01  # standard deviation of process noise
    terrain_roughness = 0.0  # flat terrain by default
    sensor_bias = 0.0  # no sensor bias by default

    # Capture current initial weights snapshot for reproducibility
    initial_weights_snapshot = [np.copy(w) for w in common_initial_weights]

    # EKF
    ekf_bounds = [ (1e-6,1e-2), (1e-5,1e-1), (0.1,10.0), (0.1,1.2) ]
    ekf_obj = make_objective('EKF', initial_weights_snapshot, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
    ekf_res = differential_evolution(ekf_obj, ekf_bounds, pop_factor=10, generations=35, seed=42)
    print(f"[DE][EKF] Best score={ekf_res['best_score']:.6e} params={ekf_res['best_params']}")

    # UKF
    ukf_bounds = [ (1e-6,1e-2), (1e-5,1e-1), (0.1,10.0), (0.1,1.2), (1e-4,0.5) ]
    ukf_obj = make_objective('UKF', initial_weights_snapshot, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
    ukf_res = differential_evolution(ukf_obj, ukf_bounds, pop_factor=10, generations=35, seed=7)
    print(f"[DE][UKF] Best score={ukf_res['best_score']:.6e} params={ukf_res['best_params']}")

    # PF (fixed n_particles=500)
    pf_bounds = [ (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (0.3,0.9) ]
    pf_obj = make_objective('PF', initial_weights_snapshot, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
    pf_res = differential_evolution(pf_obj, pf_bounds, pop_factor=12, generations=40, seed=11)
    print(f"[DE][PF ] Best score={pf_res['best_score']:.6e} params={pf_res['best_params']}")

    total_time = time.time()-start_total
    print(f"[DE] Optimization finished in {total_time:.1f}s")
else:
    ekf_res = ukf_res = pf_res = None

# Store for later use by main simulation rerun
optimized_params = {
    'EKF': ekf_res['best_params'] if RUN_DE else None,
    'UKF': ukf_res['best_params'] if RUN_DE else None,
    'PF' : pf_res['best_params'] if RUN_DE else None
}
print("Optimized parameter sets (raw):", optimized_params)


[DE] Starting hyperparameter optimization (short horizon)...
[DE][EKF] Best score=1.000000e+06 params=[3.74602665e-03 9.50719235e-02 7.34674002e+00 7.58524333e-01]
[DE][UKF] Best score=1.000000e+06 params=[7.64006585e-04 7.79940800e-02 4.44025139e+00 8.95811696e-01
 4.88996957e-01]
[DE][PF ] Best score=1.000000e+06 params=[0.18108942 0.02045577 0.46375531 0.725209   0.4207834  0.48594167
 0.30766849]
[DE] Optimization finished in 0.4s
Optimized parameter sets (raw): {'EKF': array([3.74602665e-03, 9.50719235e-02, 7.34674002e+00, 7.58524333e-01]), 'UKF': array([7.64006585e-04, 7.79940800e-02, 4.44025139e+00, 8.95811696e-01,
       4.88996957e-01]), 'PF': array([0.18108942, 0.02045577, 0.46375531, 0.725209  , 0.4207834 ,
       0.48594167, 0.30766849])}


In [8]:
# ============================================================
# 5) Simulation Main Loop (uses optimized params if present)
# ============================================================

# --- Simulation settings ---
n_steps = 1500
dt = 0.02
t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

process_noise_type = 'mixed'  # 'mixed' | 'laplacian' | 'uniform' | 'gaussian'
process_noise_std = 0.01
terrain_roughness = 0.01
sensor_bias = [0.001, 0.0005, 0.001]  # Small systematic biases [x, y, theta]

# --- True system init ---
x_true = np.zeros((n_steps, 3))
x_true[0] = [0.0, 0.0, 0.0]  # Initial conditions for mobile robot [x, y, theta]

# --- Control trajectory ---
trajectory_type = 'figure8'  # 'straight', 'circle', 'figure8', 'mixed', 'obstacle_avoidance'

# --- RHONN config ---
num_neurons = 3  # Three states for mobile robot [x, y, theta]
num_features = 17  # Feature vector size for 3 states + controls
num_weights_per_neuron = num_features

# --- Common initial weights for fair comparison ---
common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
print("Common Initial Weights:")
for i, w in enumerate(common_initial_weights):
    print(f"  Neuron {i}: {w}")

# Extract optimized parameters if available
opt_EKF = optimized_params.get('EKF') if 'optimized_params' in globals() else None
opt_UKF = optimized_params.get('UKF') if 'optimized_params' in globals() else None
opt_PF  = optimized_params.get('PF')  if 'optimized_params' in globals() else None

# Fallback defaults
if opt_EKF is None:
    opt_EKF = [2e-4, 8e-3, 1.5, 0.4]
if opt_UKF is None:
    opt_UKF = [2e-4, 8e-3, 1.5, 0.6, 1e-2]
if opt_PF is None:
    # Map to Qx,Qy,Qth,Rx,Ry,Rth,ess_ratio
    opt_PF = [0.05,0.05,0.6, 0.05,0.05,0.6, 0.5]

print("\nUsing parameter sets:")
print(f"EKF -> Q_init={opt_EKF[0]:.3e} R_init={opt_EKF[1]:.3e} P_init={opt_EKF[2]:.3f} eta={opt_EKF[3]:.3f}")
print(f"UKF -> Q_init={opt_UKF[0]:.3e} R_init={opt_UKF[1]:.3e} P_init={opt_UKF[2]:.3f} eta={opt_UKF[3]:.3f} alpha={opt_UKF[4]:.3e}")
print(f"PF  -> Q=[{opt_PF[0]:.3f},{opt_PF[1]:.3f},{opt_PF[2]:.3f}] R=[{opt_PF[3]:.3f},{opt_PF[4]:.3f},{opt_PF[5]:.3f}] ESS_ratio={opt_PF[6]:.2f}")

# --- EKF ---
ekf_trainer = EKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=opt_EKF[0], R_init=opt_EKF[1], P_init=opt_EKF[2], eta=opt_EKF[3]
)
x_hat_ekf = np.zeros((n_steps, 3))
x_hat_ekf[0] = x_true[0]

# --- UKF ---
ukf_trainer = UKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=opt_UKF[0], R_init=opt_UKF[1], P_init=opt_UKF[2], eta=opt_UKF[3],
    alpha=opt_UKF[4], beta=2.0
)
x_hat_ukf = np.zeros((n_steps, 3))
x_hat_ukf[0] = x_true[0]

# --- PF --- (n_particles fixed at 500)
n_particles = 500
Q_std_per_state = opt_PF[0:3]
R_std_per_state = opt_PF[3:6]
ess_threshold = n_particles * opt_PF[6]

pf_trainer = PF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=n_particles,
    initial_weights=common_initial_weights,
    Q_std=Q_std_per_state, R_std=R_std_per_state,
    ess_threshold=ess_threshold
)
# Force identical particle initialization
for i in range(num_neurons):
    pf_trainer.particles[i] = np.tile(common_initial_weights[i], (pf_trainer.n_particles,1))
    pf_trainer.weights_pf[i] = np.ones(pf_trainer.n_particles)/pf_trainer.n_particles

x_hat_pf = np.zeros((n_steps, 3))
x_hat_pf[0] = x_true[0]

print("\nStarting mobile robot simulation (optimized params)...")
for k in range(n_steps - 1):
    t_current = k * dt
    u_current = generate_realistic_trajectory(t_current, trajectory_type)
    x_true[k+1] = plant(x_true[k], u_current, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)

    # EKF
    ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ekf[k], x_hat_previous=x_hat_ekf[k], u_input=u_current)
    x_state_for_z_ekf = np.copy(x_hat_ekf[k])
    x_state_for_z_ekf[0] = x_hat_ekf[k][0]
    x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0], u_current)
    x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1], u_current)
    x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2], u_current)

    # UKF
    ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ukf[k], x_hat_previous=x_hat_ukf[k], u_input=u_current)
    x_state_for_z_ukf = np.copy(x_hat_ukf[k])
    x_state_for_z_ukf[0] = x_hat_ukf[k][0]
    x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0], u_current)
    x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1], u_current)
    x_hat_ukf[k+1, 2] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[2], u_current)

    # PF
    pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_pf[k], x_hat_previous=x_hat_pf[k], u_input=u_current)
    pf_weight_estimates = pf_trainer.get_estimate()
    x_state_for_z_pf = np.copy(x_hat_pf[k])
    x_state_for_z_pf[0] = x_hat_pf[k][0]
    x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0], u_current)
    x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1], u_current)
    x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2], u_current)

    if k % (n_steps // 10) == 0:
        print(f"Simulation progress: {k/n_steps*100:.1f}%")

print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [-0.15005926  0.46975734  0.00835211 -0.47687371  0.02216165 -0.49256089
  0.31889399 -0.35990911  0.07328857  0.07855174 -0.03174281  0.1127649
  0.28661564 -0.48104635  0.14146386 -0.06071363 -0.12295845]
  Neuron 1: [ 0.4731365   0.15490477 -0.06802077  0.45459086  0.23974714 -0.31454769
  0.44925656  0.28508069  0.28706442  0.07490726  0.29980899  0.27239784
 -0.05627435 -0.35448292  0.41363992  0.10652348  0.26231813]
  Neuron 2: [-0.29986569 -0.32989005  0.43663333  0.30339997  0.15665366 -0.06204716
  0.18255439  0.38937393 -0.07103118 -0.17744585  0.32474943 -0.47261113
 -0.46460615 -0.45700385  0.16092255 -0.23867049 -0.28675658]

Using parameter sets:
EKF -> Q_init=3.746e-03 R_init=9.507e-02 P_init=7.347 eta=0.759
UKF -> Q_init=7.640e-04 R_init=7.799e-02 P_init=4.440 eta=0.896 alpha=4.890e-01
PF  -> Q=[0.181,0.020,0.464] R=[0.725,0.421,0.486] ESS_ratio=0.31

Starting mobile robot simulation (optimized params)...
Simulation progress: 0.0%
Si

In [9]:
# ============================================================
# 6) Results & plots for Differential Drive Mobile Robot
# ============================================================

mse_x_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
mse_y_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
mse_theta_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)

mse_x_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
mse_y_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
mse_theta_ukf = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2)

mse_x_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
mse_y_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)
mse_theta_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)

print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal UKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ukf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print("\n--- Performance Comparison (MSE) ---")
print(f"EKF MSE x:     {mse_x_ekf:.6f}")
print(f"EKF MSE y:     {mse_y_ekf:.6f}")
print(f"EKF MSE theta: {mse_theta_ekf:.6f}")
print(f"UKF MSE x:     {mse_x_ukf:.6f}")
print(f"UKF MSE y:     {mse_y_ukf:.6f}")
print(f"UKF MSE theta: {mse_theta_ukf:.6f}")
print(f"PF  MSE x:     {mse_x_pf:.6f}")
print(f"PF  MSE y:     {mse_y_pf:.6f}")
print(f"PF  MSE theta: {mse_theta_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'x', 'desc': 'X Position', 'y_label': 'X Position (m)', 'chi': 'χₓ (True X)', 'x': 'X (Est.)'},
    {'idx': 1, 'var': 'y', 'desc': 'Y Position', 'y_label': 'Y Position (m)', 'chi': 'χᵧ (True Y)', 'x': 'Y (Est.)'},
    {'idx': 2, 'var': 'theta', 'desc': 'Orientation', 'y_label': 'Orientation (rad)', 'chi': 'χθ (True θ)', 'x': 'θ (Est.)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines', name=state_info['chi'], line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines', name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='blue'))
    trace_ukf = go.Scatter(x=t_history, y=x_hat_ukf[:, i], mode='lines', name=f"{state_info['x']} (UKF)", line=dict(dash='dashdot', color='green'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines', name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='red'))
    fig = go.Figure([trace_plant, trace_ekf, trace_ukf, trace_pf])
    fig.update_layout(title=f'Mobile Robot RHONN Identification - {state_info["var"]}', xaxis_title='Time (s)', yaxis_title=state_info['y_label'], legend=dict(x=0, y=1, orientation='h'), font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white')
    fig.show()

error_x_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_y_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_theta_ekf = x_true[:, 2] - x_hat_ekf[:, 2]
error_x_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_y_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
error_theta_ukf = x_true[:, 2] - x_hat_ukf[:, 2]
error_x_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_y_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_theta_pf = x_true[:, 2] - x_hat_pf[:, 2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ekf, mode='lines', name=f'EKF Err X ({mse_x_ekf:.2e})', opacity=0.7, line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ukf, mode='lines', name=f'UKF Err X ({mse_x_ukf:.2e})', opacity=0.7, line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_pf, mode='lines', name=f'PF Err X ({mse_x_pf:.2e})', opacity=0.7, line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ekf, mode='lines', name=f'EKF Err Y ({mse_y_ekf:.2e})', opacity=0.7, line=dict(color='blue', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ukf, mode='lines', name=f'UKF Err Y ({mse_y_ukf:.2e})', opacity=0.7, line=dict(color='green', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_pf, mode='lines', name=f'PF Err Y ({mse_y_pf:.2e})', opacity=0.7, line=dict(color='red', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ekf, mode='lines', name=f'EKF Err θ ({mse_theta_ekf:.2e})', opacity=0.7, line=dict(color='blue', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ukf, mode='lines', name=f'UKF Err θ ({mse_theta_ukf:.2e})', opacity=0.7, line=dict(color='green', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_pf, mode='lines', name=f'PF Err θ ({mse_theta_pf:.2e})', opacity=0.7, line=dict(color='red', dash='dash')))
fig2.update_layout(title='Identification Errors (MSE values)', xaxis_title='Time (s)', yaxis_title='Error', legend=dict(x=0, y=1, orientation='h'), font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white')
fig2.show()

# 2D Trajectory plot
fig_trajectory = go.Figure()
fig_trajectory.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1], mode='lines', name='True', line=dict(color='black', width=3)))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], mode='lines', name='EKF', line=dict(color='blue', width=2, dash='dash')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], mode='lines', name='UKF', line=dict(color='green', width=2, dash='dashdot')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], mode='lines', name='PF', line=dict(color='red', width=2, dash='dot')))
fig_trajectory.add_trace(go.Scatter(x=[x_true[0, 0]], y=[x_true[0, 1]], mode='markers', name='Start', marker=dict(color='green', size=10, symbol='star')))
fig_trajectory.add_trace(go.Scatter(x=[x_true[-1, 0]], y=[x_true[-1, 1]], mode='markers', name='End', marker=dict(color='red', size=10, symbol='square')))
fig_trajectory.update_layout(title='Trajectory Comparison (Optimized Params)', xaxis_title='X (m)', yaxis_title='Y (m)', font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white', showlegend=True)
fig_trajectory.show()

mse_total_ekf = mse_x_ekf + mse_y_ekf + mse_theta_ekf
mse_total_ukf = mse_x_ukf + mse_y_ukf + mse_theta_ukf
mse_total_pf = mse_x_pf + mse_y_pf + mse_theta_pf
mse_totals = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf}
best_filter = min(mse_totals, key=mse_totals.get)
print(f"\nBest overall performance: {best_filter} (total MSE: {mse_totals[best_filter]:.6f})")

# --- Parameter summary ---
print("\n--- Optimized Parameter Summary ---")
print(f"EKF params: Q={ekf_trainer.Q[0][0,0]:.3e} R={ekf_trainer.R[0][0]:.3e} P0~{ekf_trainer.P[0][0,0]:.3e} eta={ekf_trainer.eta:.3f}")
print(f"UKF params: alpha={ukf_trainer.alpha:.3e} eta={ukf_trainer.eta:.3f} Qdiag={ukf_trainer.Q[0][0,0]:.3e} R={ukf_trainer.R[0][0]:.3e}")
print(f"PF params: Q_std={pf_trainer.Q_std} R_std={pf_trainer.R_std} ESS_th={pf_trainer.ess_threshold:.1f} n_particles={pf_trainer.n_particles}")

print("Optimization + simulation complete.")


Final EKF-RHONN Weights:
  Neuron 1 (x): [-0.0024546   0.48455852 -0.02371828 -0.30667317 -0.12144272 -0.43352731
  0.3904448  -0.33426468  0.01360906  0.00731098 -0.01111594  0.35784235
 -0.11493332 -0.51622771  0.81357429  0.02167524 -0.24900987]
  Neuron 2 (y): [ 0.01735072  0.03752124 -0.21522997  0.29455874 -0.12042176 -0.36081405
  0.02143423  0.09933134  0.23608322  0.04292751  0.00714007 -0.30450583
 -0.00700851 -0.49094383  0.01199341  0.87847462 -0.14732677]
  Neuron 3 (theta): [-1.58947447 -0.83913166  1.45141209 -0.02295479  0.95192129 -0.42994707
 -2.80161378  1.18261815  3.17519074  0.72111736 -0.25502926 -1.30803577
 -1.31454451 -1.15041193  0.49794655 -0.4971809  -1.37255408]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 1.92521108e-01  8.23757060e-02 -3.63716999e-01 -4.92032335e-01
 -1.16607866e-01 -5.01678308e-01  2.42545096e-01 -4.41130197e-01
 -5.42200989e-02  2.88911744e-01  4.12764311e-04 -3.56938429e-01
 -1.93875306e-02 -6.29645788e-01  4.39405137e-01  3.19832939e


Best overall performance: UKF (total MSE: 0.090819)

--- Optimized Parameter Summary ---
EKF params: Q=3.746e-03 R=9.507e-02 P0~1.120e+01 eta=0.759
UKF params: alpha=4.890e-01 eta=0.896 Qdiag=7.640e-04 R=7.799e-02
PF params: Q_std=[0.18108941918789243, 0.02045576624613696, 0.4637553079718463] R_std=[0.7252089952629557, 0.4207834009831397, 0.4859416710696146] ESS_th=153.8 n_particles=500
Optimization + simulation complete.
